# Structured Output Pipelines: Schema Validation and Grammar-Constrained Generation

Unstructured text is hard to reliably parse in downstream systems. Structured output — JSON, XML, or other schema-compliant formats — makes LLM outputs composable with the rest of your data pipeline.

In [ ]:
from dataclasses import dataclass
from typing import Callable
import json, re

@dataclass
class StructuredOutputSchema:
    name: str
    schema: dict
    strict: bool = True

    def validate(self, data: dict) -> tuple:
        errors = []
        required = self.schema.get('required', [])
        properties = self.schema.get('properties', {})
        for field in required:
            if field not in data:
                errors.append(f'Missing required field: {field}')
        for k, v in data.items():
            if k not in properties:
                if self.strict:
                    errors.append(f'Unexpected field: {k}')
                continue
            expected_type = properties[k].get('type')
            if expected_type == 'number' and not isinstance(v, (int, float)):
                errors.append(f'{k} must be a number')
            elif expected_type == 'string' and not isinstance(v, str):
                errors.append(f'{k} must be a string')
            elif expected_type == 'boolean' and not isinstance(v, bool):
                errors.append(f'{k} must be a boolean')
            if 'enum' in properties[k] and v not in properties[k]['enum']:
                errors.append(f'{k} must be one of {properties[k]["enum"]}')
        return len(errors) == 0, errors

class StructuredOutputPipeline:
    def __init__(self, model_fn: Callable, schema: StructuredOutputSchema, max_retries: int = 3):
        self.model = model_fn
        self.schema = schema
        self.max_retries = max_retries

    def _extract_json(self, text: str) -> dict:
        json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
        return json.loads(text.strip())

    def run(self, prompt: str) -> dict:
        schema_str = json.dumps(self.schema.schema, indent=2)
        full_prompt = f'{prompt}\n\nRespond with JSON matching this schema:\n{schema_str}'
        for attempt in range(self.max_retries):
            response = self.model(full_prompt)
            try:
                data = self._extract_json(response)
                valid, errors = self.schema.validate(data)
                if valid:
                    return {'success': True, 'data': data, 'attempt': attempt+1}
                # Fix prompt with errors for next attempt
                full_prompt = f'{prompt}\n\nPrevious errors: {errors}\nSchema:\n{schema_str}'
            except json.JSONDecodeError as e:
                full_prompt = f'{prompt}\n\nPrevious response was not valid JSON: {e}\nSchema:\n{schema_str}'
        return {'success': False, 'data': None, 'attempt': self.max_retries}

sentiment_schema = StructuredOutputSchema(
    name='sentiment_analysis',
    schema={
        'type': 'object',
        'properties': {
            'sentiment': {'type': 'string', 'enum': ['positive', 'negative', 'neutral']},
            'confidence': {'type': 'number'},
            'key_phrases': {'type': 'array'},
        },
        'required': ['sentiment', 'confidence'],
    }
)

attempt_n = [0]
def mock_model(prompt):
    attempt_n[0] += 1
    if attempt_n[0] == 1:
        return 'The sentiment is positive, confidence 0.9'  # not JSON
    return '{"sentiment": "positive", "confidence": 0.9, "key_phrases": ["great", "helpful"]}'

pipeline = StructuredOutputPipeline(mock_model, sentiment_schema)
result = pipeline.run('Analyze sentiment: This product is great and very helpful!')
print(f'Success: {result["success"]}')
print(f'Attempts: {result["attempt"]}')
print(f'Data: {result["data"]}')

## Grammar-Constrained Generation

For strict output requirements, grammar-constrained decoding (used in llama.cpp, vLLM, outlines library) modifies the token sampling to only allow tokens that keep the output valid according to a formal grammar or JSON schema.

Advantages:
- Zero retry overhead — first generation is always schema-valid
- No possibility of JSON decode errors
- Works with any JSON schema including nested objects and arrays

Limitations:
- Slightly reduces quality by constraining token choices
- Requires access to the model's logits (not available via API)
- Complex schemas can significantly slow down generation

For API-based deployments, structured output via system prompt + retry is the practical approach. For self-hosted inference, grammar-constrained decoding is worth the engineering investment.